# 🎓 Student Performance & Academic Analytics System
### Interactive Exploratory Data Analysis & Week 3–4 Concepts Walkthrough

Welcome to the **Student Performance & Academic Analytics System** notebook. This notebook provides an end-to-end interactive data science workflow covering:
- **Python Core**: Functions, List Comprehensions, Dictionary Comprehensions, Object-Oriented Programming (OOP)
- **NumPy**: Vectorized operations, array statistics, broadcasting, correlations
- **Pandas**: Data ingestion, inspection, cleaning, imputation, merging, `groupby()`, `agg()`, filtering, and sorting
- **Matplotlib & Seaborn**: Statistical visualizations and polished academic charts

## 1. Environment Setup & Library Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project src is accessible
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.analytics import StudentAnalytics, load_data, clean_data, merge_department_data, calculate_weighted_score, categorize_students, calculate_statistics

# Global styling configuration
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams['figure.figsize'] = (10, 5)
print("Libraries successfully imported!")

## 2. Data Loading & Initial Inspection (Pandas)
We load two datasets:
1. `data/student_performance.csv` (Cohort exam scores, study hours, attendance, GPA)
2. `data/department_info.csv` (Department names, faculty, specialization)

In [2]:
student_path = os.path.join(project_root, "data", "student_performance.csv")
dept_path = os.path.join(project_root, "data", "department_info.csv")

student_df, dept_df = load_data(student_path, dept_path)

print("--- Student Dataset Shape ---", student_df.shape)
display(student_df.head())

print("--- Department Info ---")
display(dept_df)

## 3. Data Cleaning & Preprocessing (Pandas & NumPy)
Data cleaning is crucial for robust analytics. We perform:
1. **Deduplication**: Detecting and dropping duplicate student records.
2. **Missing Value Imputation**: Detecting null values and filling them with mean values.
3. **Type Standardization**: Ensuring numeric columns have appropriate float/int types.

In [3]:
# Inspect missing values before cleaning
print("Missing values per column before cleaning:")
print(student_df.isnull().sum())

# Inspect duplicate rows
print("\nDuplicate student records:", student_df.duplicated(subset=['Student_ID']).sum())

# Apply cleaning function
cleaned_df, summary = clean_data(student_df)
print("\nCleaning Summary:", summary)
print("Missing values after cleaning:", cleaned_df.isnull().sum().sum())

## 4. Dataset Merging (`pd.merge()`)
We join student performance records with the department metadata using `pd.merge()` on the `Department` key.

In [4]:
merged_df = merge_department_data(cleaned_df, dept_df)
print("Merged DataFrame preview:")
display(merged_df[['Student_ID', 'Name', 'Department', 'Department_Name', 'Faculty', 'Final_Score']].head())

## 5. NumPy Vectorization: Weighted Score Calculation
We compute the overall **Final Weighted Score** using NumPy vectorization and broadcasting:
$$\text{Weighted Score} = 0.30 \times \text{Assignment} + 0.30 \times \text{Midterm} + 0.40 \times \text{Final Exam}$$

In [5]:
weighted_df = calculate_weighted_score(merged_df)
print("Vectorized Weighted Scores computed successfully:")
display(weighted_df[['Student_ID', 'Assignment_Score', 'Midterm_Score', 'Final_Score', 'Weighted_Score']].head())

## 6. Categorizing Students into Academic Tiers
Using predefined performance criteria, we segment students:
- **Excellent**: Weighted Score $\ge 85$
- **Good**: $70 \le \text{Weighted Score} < 85$
- **Average**: $50 \le \text{Weighted Score} < 70$
- **At Risk**: Weighted Score $< 50$

In [6]:
final_df = categorize_students(weighted_df)
print("Performance Category Distribution:")
print(final_df['Performance_Category'].value_counts())

## 7. Statistical Analysis & Addressing the 8 Core Questions

In [7]:
stats = calculate_statistics(final_df)
corr = stats['correlations']
top_10 = stats['top_10_students_df']

print("=" * 75)
print("ANSWERS TO THE 8 CORE RESEARCH QUESTIONS:")
print("=" * 75)
print(f"Q1: Overall Average Final Score       : {stats['avg_final_score']:.2f}")
print(f"Q2: Highest Scoring Department        : {stats['dept_highest_score']['dept']} ({stats['dept_highest_score']['avg_score']:.2f})")
print(f"Q3: Attendance Impact on Final Score   : Positive (r = {corr['attendance_vs_final']:+.4f})")
print(f"Q4: Study Hours Impact on Final Score  : Strong Positive (r = {corr['study_hours_vs_final']:+.4f})")
print(f"Q5: Percentage of At-Risk Students     : {stats['at_risk_percentage']:.2f}% ({stats['at_risk_count']} students)")
print(f"Q6: Top Student by Weighted Score      : {top_10.iloc[0]['Name']} ({top_10.iloc[0]['Department']} - {top_10.iloc[0]['Weighted_Score']:.2f})")
print(f"Q7: Highest Average GPA Department     : {stats['dept_highest_gpa']['dept']} ({stats['dept_highest_gpa']['avg_gpa']:.2f})")
print(f"Q8: Correlation Summary:")
print(f"    - Attendance vs Final Score        : r = {corr['attendance_vs_final']:+.4f}")
print(f"    - Study Hours vs Final Score       : r = {corr['study_hours_vs_final']:+.4f}")
print(f"    - Previous GPA vs Final Score      : r = {corr['previous_gpa_vs_final']:+.4f}")
print("=" * 75)

print("\n--- Top 10 Students ---")
display(top_10)

## 8. Python List & Dictionary Comprehensions Demonstration
Demonstrating idiomatic Python comprehensions for concise filtering and data aggregation.

In [8]:
# List Comprehension: Filter Dean's List Honor Students (Weighted Score >= 90)
honors_list = [
    f"{row['Student_ID']} - {row['Name']} ({row['Department']}: {row['Weighted_Score']})"
    for _, row in final_df[final_df['Weighted_Score'] >= 90.0].iterrows()
]
print(f"Honors Students Count: {len(honors_list)}")
for s in honors_list[:5]:
    print("  ★", s)

# Dictionary Comprehension: Department GPA & Score Summary
dept_summary = {
    dept: {
        'Total_Students': len(final_df[final_df['Department'] == dept]),
        'Avg_Score': round(final_df[final_df['Department'] == dept]['Final_Score'].mean(), 2),
        'Avg_GPA': round(final_df[final_df['Department'] == dept]['Previous_GPA'].mean(), 2)
    }
    for dept in sorted(final_df['Department'].unique())
}
print("\nDepartment Summary (Dictionary Comprehension):")
import pprint
pprint.pprint(dept_summary)

## 9. Object-Oriented Programming (OOP) Class: `StudentAnalytics`
Encapsulating the entire analytical workflow in a cohesive class.

In [9]:
# OOP Instantiation and execution
analytics = StudentAnalytics(student_path=student_path, department_path=dept_path)
analytics.clean_data()
oop_stats = analytics.calculate_statistics()
print(f"StudentAnalytics OOP executed cleanly. Total Analyzed: {oop_stats['total_students']} students.")

## 10. Matplotlib & Seaborn Visualizations
Rendering all 7 analytical charts inline.

In [10]:
from src.visualization import (
    plot_dept_final_scores,
    plot_study_hours_vs_final_score,
    plot_attendance_vs_final_score,
    plot_final_score_distribution,
    plot_category_distribution,
    plot_correlation_matrix,
    plot_dept_score_distribution
)

temp_charts_dir = os.path.join(project_root, "output", "charts")

# 1. Dept Final Scores
plot_dept_final_scores(final_df, os.path.join(temp_charts_dir, "01_avg_final_score_by_dept.png"))

# 2. Study Hours vs Final Score
plot_study_hours_vs_final_score(final_df, os.path.join(temp_charts_dir, "02_study_hours_vs_final_score.png"))

# 3. Attendance vs Final Score
plot_attendance_vs_final_score(final_df, os.path.join(temp_charts_dir, "03_attendance_vs_final_score.png"))

# 4. Final Score Distribution
plot_final_score_distribution(final_df, os.path.join(temp_charts_dir, "04_final_score_distribution.png"))

# 5. Performance Category Distribution
plot_category_distribution(final_df, os.path.join(temp_charts_dir, "05_performance_category_distribution.png"))

# 6. Correlation Matrix Heatmap
plot_correlation_matrix(final_df, os.path.join(temp_charts_dir, "06_correlation_matrix_heatmap.png"))

# 7. Score Distribution by Department Boxplot
plot_dept_score_distribution(final_df, os.path.join(temp_charts_dir, "07_score_distribution_by_department.png"))

print("All 7 visualizations generated successfully in output/charts/")